# K-Means Clustering from Scratch

**Goal:** Implement Lloyd's algorithm, K-means++, mini-batch K-means, and cluster evaluation metrics -- all in NumPy.

---

## Core Intuition

K-means minimizes **within-cluster sum of squares (WCSS / inertia)**:

$$\min_{\{\mu_k\}} \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

**Lloyd's algorithm** alternates two steps:
1. **Assignment:** Assign each point to nearest centroid
2. **Update:** Recompute centroids as mean of assigned points

This is coordinate descent on the WCSS objective and is guaranteed to converge (but to a local minimum).

**Key insight:** K-means is equivalent to **hard EM** with spherical Gaussians of equal variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Lloyd's Algorithm (Basic K-Means)

In [ ]:
class KMeans:
    """K-means clustering using Lloyd's algorithm."""
    
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-6, init='random', random_state=42):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        self.init = init
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None
        self.n_iters_ = 0
        self.history = []  # for visualization
    
    def _init_centroids(self, X):
        rng = np.random.RandomState(self.random_state)
        if self.init == 'random':
            idx = rng.choice(X.shape[0], self.n_clusters, replace=False)
            return X[idx].copy()
        elif self.init == 'kmeans++':
            return self._kmeanspp_init(X, rng)
        else:
            raise ValueError(f"Unknown init: {self.init}")
    
    def _kmeanspp_init(self, X, rng):
        """K-means++ initialization: spread out initial centroids."""
        n_samples = X.shape[0]
        centroids = [X[rng.randint(n_samples)]]
        
        for _ in range(1, self.n_clusters):
            # Distance to nearest existing centroid
            dists = np.array([np.min(np.sum((X - c) ** 2, axis=1)) for c in centroids]).T
            if dists.ndim > 1:
                dists = np.min(dists, axis=1)
            else:
                dists = np.sum((X - centroids[0]) ** 2, axis=1)
            
            # Sample proportional to distance squared
            probs = dists / dists.sum()
            idx = rng.choice(n_samples, p=probs)
            centroids.append(X[idx])
        
        return np.array(centroids)
    
    def _assign_clusters(self, X):
        """Assign each point to nearest centroid."""
        # Vectorized distance computation
        dists = np.sum((X[:, np.newaxis] - self.centroids[np.newaxis]) ** 2, axis=2)
        return np.argmin(dists, axis=1)
    
    def _compute_inertia(self, X, labels):
        """Compute within-cluster sum of squares."""
        return sum(np.sum((X[labels == k] - self.centroids[k]) ** 2) 
                   for k in range(self.n_clusters) if np.any(labels == k))
    
    def fit(self, X):
        self.centroids = self._init_centroids(X)
        self.history = [(self.centroids.copy(), None)]
        
        for i in range(self.max_iters):
            # Assignment step
            labels = self._assign_clusters(X)
            
            # Update step
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                if np.any(labels == k):
                    new_centroids[k] = X[labels == k].mean(axis=0)
                else:
                    new_centroids[k] = self.centroids[k]  # keep old if empty
            
            self.history.append((new_centroids.copy(), labels.copy()))
            
            # Check convergence
            shift = np.sum((new_centroids - self.centroids) ** 2)
            self.centroids = new_centroids
            self.n_iters_ = i + 1
            
            if shift < self.tol:
                break
        
        self.labels_ = labels
        self.inertia_ = self._compute_inertia(X, labels)
        return self
    
    def predict(self, X):
        return self._assign_clusters(X)

# Generate blob data
from sklearn.datasets import make_blobs
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

km = KMeans(n_clusters=3, init='random')
km.fit(X_blobs)
print(f"Converged in {km.n_iters_} iterations")
print(f"Inertia: {km.inertia_:.2f}")

## 2. Visualize K-Means Iterations

In [ ]:
km_vis = KMeans(n_clusters=3, init='random', random_state=10)
km_vis.fit(X_blobs)

n_show = min(6, len(km_vis.history))
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors_list = ['#e74c3c', '#3498db', '#2ecc71']

for idx, ax in enumerate(axes.ravel()[:n_show]):
    centroids, labels = km_vis.history[idx]
    
    if labels is None:
        ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c='gray', alpha=0.3, s=15)
    else:
        for k in range(3):
            mask = labels == k
            ax.scatter(X_blobs[mask, 0], X_blobs[mask, 1], c=colors_list[k], alpha=0.4, s=15)
    
    ax.scatter(centroids[:, 0], centroids[:, 1], c=colors_list[:3], marker='X',
               s=200, edgecolors='black', linewidths=2, zorder=5)
    ax.set_title(f'Iteration {idx}' if idx > 0 else 'Initialization')
    ax.grid(True, alpha=0.3)

# Fill remaining
for idx in range(n_show, 6):
    axes.ravel()[idx].set_visible(False)

plt.suptitle('K-Means Iteration Steps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. K-Means++ vs Random Initialization

K-means++ chooses initial centroids that are spread out, proportional to $D(x)^2$ (squared distance to nearest existing centroid). This gives $O(\log k)$-competitive solution in expectation.

In [ ]:
# Compare multiple runs
inertias_random = []
inertias_pp = []

for seed in range(50):
    km_r = KMeans(n_clusters=3, init='random', random_state=seed)
    km_r.fit(X_blobs)
    inertias_random.append(km_r.inertia_)
    
    km_p = KMeans(n_clusters=3, init='kmeans++', random_state=seed)
    km_p.fit(X_blobs)
    inertias_pp.append(km_p.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(inertias_random, bins=20, alpha=0.7, color='red', label='Random init')
axes[0].hist(inertias_pp, bins=20, alpha=0.7, color='blue', label='K-means++')
axes[0].set_xlabel('Inertia')
axes[0].set_ylabel('Count')
axes[0].set_title('Inertia Distribution Over 50 Runs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].boxplot([inertias_random, inertias_pp], labels=['Random', 'K-means++'])
axes[1].set_ylabel('Inertia')
axes[1].set_title('Inertia: Random vs K-means++')
axes[1].grid(True, alpha=0.3)

print(f"Random init: mean={np.mean(inertias_random):.1f}, std={np.std(inertias_random):.1f}")
print(f"K-means++:   mean={np.mean(inertias_pp):.1f}, std={np.std(inertias_pp):.1f}")

plt.tight_layout()
plt.show()

## 4. Mini-Batch K-Means

For large datasets, use random mini-batches to update centroids. Much faster, slightly less accurate.

In [ ]:
class MiniBatchKMeans:
    """Mini-batch K-means for large-scale clustering."""
    
    def __init__(self, n_clusters=3, batch_size=50, n_iters=100, random_state=42):
        self.n_clusters = n_clusters
        self.batch_size = batch_size
        self.n_iters = n_iters
        self.random_state = random_state
        self.centroids = None
        self.counts = None  # per-centroid sample counts for averaging
    
    def fit(self, X):
        rng = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]
        
        # Initialize with k-means++
        idx = rng.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[idx].copy()
        self.counts = np.ones(self.n_clusters)
        self.inertia_history = []
        
        for t in range(self.n_iters):
            # Sample mini-batch
            batch_idx = rng.choice(n_samples, self.batch_size, replace=False)
            X_batch = X[batch_idx]
            
            # Assign
            dists = np.sum((X_batch[:, np.newaxis] - self.centroids[np.newaxis]) ** 2, axis=2)
            labels = np.argmin(dists, axis=1)
            
            # Update with learning rate
            for k in range(self.n_clusters):
                mask = labels == k
                if mask.any():
                    count = mask.sum()
                    self.counts[k] += count
                    eta = count / self.counts[k]  # decreasing learning rate
                    self.centroids[k] = (1 - eta) * self.centroids[k] + eta * X_batch[mask].mean(axis=0)
            
            # Track inertia periodically
            if t % 10 == 0:
                all_dists = np.sum((X[:, np.newaxis] - self.centroids[np.newaxis]) ** 2, axis=2)
                all_labels = np.argmin(all_dists, axis=1)
                inertia = sum(np.sum((X[all_labels == k] - self.centroids[k]) ** 2)
                              for k in range(self.n_clusters) if np.any(all_labels == k))
                self.inertia_history.append(inertia)
        
        # Final labels
        all_dists = np.sum((X[:, np.newaxis] - self.centroids[np.newaxis]) ** 2, axis=2)
        self.labels_ = np.argmin(all_dists, axis=1)
        self.inertia_ = self.inertia_history[-1] if self.inertia_history else 0
        return self
    
    def predict(self, X):
        dists = np.sum((X[:, np.newaxis] - self.centroids[np.newaxis]) ** 2, axis=2)
        return np.argmin(dists, axis=1)

# Compare
import time

X_large, _ = make_blobs(n_samples=5000, centers=5, cluster_std=1.0, random_state=42)

t0 = time.time()
km_full = KMeans(n_clusters=5, init='kmeans++').fit(X_large)
t_full = time.time() - t0

t0 = time.time()
km_mini = MiniBatchKMeans(n_clusters=5, batch_size=100, n_iters=100).fit(X_large)
t_mini = time.time() - t0

print(f"Full K-means: inertia={km_full.inertia_:.1f}, time={t_full:.3f}s")
print(f"Mini-batch:   inertia={km_mini.inertia_:.1f}, time={t_mini:.3f}s")

## 5. Elbow Method and Silhouette Score

In [ ]:
def silhouette_score(X, labels):
    """Compute mean silhouette score.
    
    For each point i:
      a(i) = mean distance to other points in same cluster
      b(i) = mean distance to points in nearest different cluster
      s(i) = (b(i) - a(i)) / max(a(i), b(i))
    """
    n = X.shape[0]
    unique_labels = np.unique(labels)
    
    if len(unique_labels) <= 1:
        return 0.0
    
    # Pairwise distances
    dists = np.sqrt(np.sum((X[:, np.newaxis] - X[np.newaxis]) ** 2, axis=2))
    
    silhouettes = np.zeros(n)
    
    for i in range(n):
        # a(i): mean intra-cluster distance
        same_cluster = labels == labels[i]
        same_cluster[i] = False
        if same_cluster.sum() > 0:
            a_i = np.mean(dists[i, same_cluster])
        else:
            a_i = 0.0
        
        # b(i): min of mean inter-cluster distances
        b_i = np.inf
        for k in unique_labels:
            if k == labels[i]:
                continue
            other = labels == k
            if other.sum() > 0:
                b_i = min(b_i, np.mean(dists[i, other]))
        
        silhouettes[i] = (b_i - a_i) / max(a_i, b_i) if max(a_i, b_i) > 0 else 0
    
    return np.mean(silhouettes), silhouettes

# Elbow method + Silhouette
K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km_temp = KMeans(n_clusters=k, init='kmeans++')
    km_temp.fit(X_blobs)
    inertias.append(km_temp.inertia_)
    sil_mean, _ = silhouette_score(X_blobs, km_temp.labels_)
    silhouettes.append(sil_mean)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='True K=3')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='True K=3')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best K by silhouette: {list(K_range)[np.argmax(silhouettes)]}")

## 6. Voronoi Regions

In [ ]:
km_3 = KMeans(n_clusters=3, init='kmeans++')
km_3.fit(X_blobs)

# Create grid for Voronoi
h = 0.1
x_min, x_max = X_blobs[:, 0].min() - 2, X_blobs[:, 0].max() + 2
y_min, y_max = X_blobs[:, 1].min() - 2, X_blobs[:, 1].max() + 2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = km_3.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='Set1')
plt.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.3)

for k in range(3):
    mask = km_3.labels_ == k
    plt.scatter(X_blobs[mask, 0], X_blobs[mask, 1], s=20, alpha=0.6, label=f'Cluster {k}')

plt.scatter(km_3.centroids[:, 0], km_3.centroids[:, 1], c='black', marker='X', s=200,
            edgecolors='white', linewidths=2, zorder=5, label='Centroids')
plt.title('K-Means Voronoi Regions')
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Failure Cases: Where K-Means Struggles

In [ ]:
from sklearn.datasets import make_moons, make_circles

# Generate challenging datasets
datasets = [
    ('Blobs (easy)', make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)),
    ('Moons (non-convex)', make_moons(n_samples=300, noise=0.1, random_state=42)),
    ('Circles (non-convex)', make_circles(n_samples=300, noise=0.05, factor=0.3, random_state=42)),
    ('Anisotropic', None),  # will generate below
]

# Anisotropic blobs
np.random.seed(42)
X_aniso, y_aniso = make_blobs(n_samples=300, centers=3, random_state=170)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = X_aniso @ transformation
datasets[3] = ('Anisotropic', (X_aniso, y_aniso))

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for col, (name, (X_data, y_data)) in enumerate(datasets):
    k = len(np.unique(y_data))
    
    # Ground truth
    axes[0, col].scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='Set1', alpha=0.6, s=15)
    axes[0, col].set_title(f'{name}\nGround Truth')
    axes[0, col].grid(True, alpha=0.3)
    
    # K-means result
    km_temp = KMeans(n_clusters=k, init='kmeans++')
    km_temp.fit(X_data)
    axes[1, col].scatter(X_data[:, 0], X_data[:, 1], c=km_temp.labels_, cmap='Set1', alpha=0.6, s=15)
    axes[1, col].scatter(km_temp.centroids[:, 0], km_temp.centroids[:, 1],
                         c='black', marker='X', s=150, edgecolors='white', linewidths=2)
    sil, _ = silhouette_score(X_data, km_temp.labels_)
    axes[1, col].set_title(f'K-Means (sil={sil:.2f})')
    axes[1, col].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Ground Truth', fontsize=12)
axes[1, 0].set_ylabel('K-Means', fontsize=12)
plt.suptitle('K-Means Success and Failure Cases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Effect of Different Scales

In [ ]:
# Data with very different feature scales
X_scaled, y_scaled = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)
X_scaled[:, 1] *= 20  # stretch one dimension

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without scaling
km_no_scale = KMeans(n_clusters=3, init='kmeans++').fit(X_scaled)
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=km_no_scale.labels_, cmap='Set1', alpha=0.6, s=15)
axes[0].set_title('Without Scaling (wrong clusters)')
axes[0].grid(True, alpha=0.3)

# With scaling
X_normed = (X_scaled - X_scaled.mean(axis=0)) / X_scaled.std(axis=0)
km_scale = KMeans(n_clusters=3, init='kmeans++').fit(X_normed)
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=km_scale.labels_, cmap='Set1', alpha=0.6, s=15)
axes[1].set_title('With Scaling (correct clusters)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Importance of Feature Scaling for K-Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Compare with sklearn

In [ ]:
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.metrics import adjusted_rand_score

our_km = KMeans(n_clusters=3, init='kmeans++', random_state=42)
our_km.fit(X_blobs)

sk_km = SklearnKMeans(n_clusters=3, init='k-means++', n_init=1, random_state=42)
sk_km.fit(X_blobs)

print(f"Our inertia:     {our_km.inertia_:.2f}")
print(f"sklearn inertia: {sk_km.inertia_:.2f}")
print(f"\nOur ARI vs true:     {adjusted_rand_score(y_blobs, our_km.labels_):.4f}")
print(f"sklearn ARI vs true: {adjusted_rand_score(y_blobs, sk_km.labels_):.4f}")

---

## Interview Questions

### "K-means convergence guarantee?"
K-means is guaranteed to converge because: (1) the assignment step never increases WCSS, (2) the update step never increases WCSS, (3) there are finitely many partitions. But it converges to a **local minimum**, not necessarily the global optimum. Run multiple times with different initializations.

### "Time complexity?"
- Per iteration: $O(nkd)$ where $n$ = samples, $k$ = clusters, $d$ = dimensions
- Total: $O(nkdt)$ where $t$ = iterations (typically small, 10-50)
- K-means++ initialization: $O(nk)$ additional
- Finding the global optimum is NP-hard

### "K-means vs GMM?"
- K-means: hard assignment, assumes spherical clusters of equal size
- GMM: soft assignment (probabilities), can model elliptical clusters of different sizes
- K-means = hard EM with uniform spherical Gaussians
- K-means is faster; GMM gives richer model

### "How to handle different cluster sizes?"
- K-means tends to split large clusters and merge small ones (Voronoi cells are equal-sized)
- Solutions: GMM (models size explicitly), DBSCAN (density-based), hierarchical clustering
- For different densities: DBSCAN, HDBSCAN, or spectral clustering

### Notes
- **K-means = hard EM:** K-means is EM for a mixture of isotropic Gaussians with equal covariance, where the E-step uses hard assignments instead of soft responsibilities.
- **K-medoids:** Uses actual data points as centers instead of means. More robust to outliers. Uses Manhattan or arbitrary distance metrics (not just Euclidean).

In [ ]:
print("Notebook 11 complete: K-Means from Scratch")
print("="*50)
print("Implemented:")
print("  - Lloyd's algorithm (basic K-means)")
print("  - K-means++ initialization")
print("  - Mini-batch K-means")
print("  - Elbow method")
print("  - Silhouette score")
print("  - Voronoi regions visualization")
print("  - Failure case analysis")
print("  - Validated against sklearn")